In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [3]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 6, x.day-1, tzinfo=timezone)    
#     utc_to = datetime(x.year, x.month, x.day, tzinfo=timezone)
#     utc_to = datetime(x.year, x.month+1, 9, tzinfo=timezone)
    utc_to = datetime(x.year, x.month, x.day+1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M30, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
#     rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=20).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [4]:
def get_rsi(close, lookback):
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     rsi_df = rsi_df.dropna()
    return rsi_df

# ibm['rsi_14'] = get_rsi(ibm['close'], 14)
# ibm = ibm.dropna()


In [5]:
def slope(x1, y1, x2, y2):
    return (y2-y1)/(x2-x1)

In [6]:
def go(a):
    g = []
    for i in range(0, len(a)):
        if a.iloc[i].rsi >= 60.0:
            g.append(slope(0, a.iloc[i-5].rsi, 5, a.iloc[i].rsi))
        else:
            g.append(0)
    return g

In [9]:
symbol = "CADJPY"
a= get_values(symbol)
print(a.tail())
a['rsi'] = get_rsi(a['close'], 40)
a['smaL']= a['rsi'].rolling(window=2).mean()
a['slope'] = go(a)
a = a.dropna()
a = a[100:]
a

                       open   close
time                               
2021-08-10 18:14:00  88.166  88.160
2021-08-10 18:15:00  88.161  88.156
2021-08-10 18:16:00  88.156  88.164
2021-08-10 18:17:00  88.165  88.169
2021-08-10 18:18:00  88.168  88.166


,open,close,rsi,smaL,slope
time,,,,,
2021-08-09 01:50:00,87.711,87.701,43.186596,43.407159,0.0
2021-08-09 01:51:00,87.700,87.682,40.524943,41.855770,0.0
2021-08-09 01:52:00,87.687,87.683,40.722157,40.623550,0.0
2021-08-09 01:53:00,87.683,87.692,42.482666,41.602411,0.0
2021-08-09 01:54:00,87.689,87.701,44.182900,43.332783,0.0
...,...,...,...,...,...
2021-08-10 18:14:00,88.166,88.160,50.667907,50.916908,0.0
2021-08-10 18:15:00,88.161,88.156,50.266474,50.467191,0.0
2021-08-10 18:16:00,88.156,88.164,51.061815,50.664145,0.0


In [10]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if a.iloc[i].rsi >= 41.0:
        if a.iloc[i].slope > 1.9 and a.iloc[i].slope > 0.0 and a.iloc[i-1].slope == 0.0 \
            and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("RSI   SLOPE")
            print(f"{round(a.iloc[i].rsi,2)}   {round(a.iloc[i].slope,2)}")
#             print("SMA   SMAH")
#             print(f"{peck}   {up}")
# #             print(f"{round(a.iloc[i].sma,6)}   {round(a.iloc[i].smaH,6)}")
#             print("CLOSE")
#             print(f"{a.iloc[i].Close}")
#             print(a.iloc[i].smaL)
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(f"{pp}---{round(a.iloc[i].rsi, 2)}--{round(a.iloc[i].slope, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
            if pp < -3.0:
                profit.append(pp)
                check = 0
#             if a.iloc[i].rsi >= 49.0:
#                 if a.iloc[i].smaL >= a.iloc[i-1].smaL:
#                     pass
#                 else:
#                     profit.append(pp)
#                     check = 0
#             if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
#                 profit.append(pp)
#                 check = 0
            if a.iloc[i].rsi <= 41.0:
                profit.append(pp)
                check = 0
#             if a.iloc[i].rsi <= 40.0:
#                 profit.append(pp)
#                 check = 0

####################
2021-08-09 10:01:00
RSI   SLOPE
63.57   3.01
********************
-0.31---65.66--3.19---87.762--2021-08-09 10:02:00
-0.52---67.03--3.7---87.774--2021-08-09 10:03:00
-0.71---68.12--3.5---87.784--2021-08-09 10:04:00
-0.34---63.8--2.5---87.764--2021-08-09 10:05:00
-0.14---61.6---0.39---87.753--2021-08-09 10:06:00
-0.45---63.59---0.41---87.77--2021-08-09 10:07:00
-1.16---67.55--0.1---87.809--2021-08-09 10:08:00
-1.19---67.74---0.08---87.811--2021-08-09 10:09:00
-1.5---69.26--1.09---87.828--2021-08-09 10:10:00
-1.23---66.42--0.96---87.813--2021-08-09 10:11:00
-0.87---62.89---0.14---87.793--2021-08-09 10:12:00
-0.72---61.55---1.2---87.785--2021-08-09 10:13:00
-0.65---60.89---1.37---87.781--2021-08-09 10:14:00
-0.76---61.53---1.55---87.787--2021-08-09 10:15:00
-0.58---59.85--0.0---87.777--2021-08-09 10:16:00
-0.49---59.03--0.0---87.772--2021-08-09 10:17:00
-0.74---60.59---0.19---87.786--2021-08-09 10:18:00
-1.18---63.06--0.43---87.81--2021-08-09 10:19:00
-0.85---60.16---0

-1.65---56.21--0.0---87.836--2021-08-09 13:17:00
-1.43---54.33--0.0---87.824--2021-08-09 13:18:00
-1.23---52.67--0.0---87.813--2021-08-09 13:19:00
-1.34---53.46--0.0---87.819--2021-08-09 13:20:00
-1.38---53.73--0.0---87.821--2021-08-09 13:21:00
-1.5---54.66--0.0---87.828--2021-08-09 13:22:00
-1.32---53.1--0.0---87.818--2021-08-09 13:23:00
-1.3599999999999999---53.37--0.0---87.82--2021-08-09 13:24:00
-1.28---52.74--0.0---87.816--2021-08-09 13:25:00
-1.45---53.99--0.0---87.825--2021-08-09 13:26:00
-1.18---51.65--0.0---87.81--2021-08-09 13:27:00
-1.18---51.65--0.0---87.81--2021-08-09 13:28:00
-0.83---48.83--0.0---87.791--2021-08-09 13:29:00
-0.72---47.98--0.0---87.785--2021-08-09 13:30:00
-1.05---50.62--0.0---87.803--2021-08-09 13:31:00
-0.85---49.06--0.0---87.792--2021-08-09 13:32:00
-0.85---49.06--0.0---87.792--2021-08-09 13:33:00
-1.09---50.94--0.0---87.805--2021-08-09 13:34:00
-0.94---49.78--0.0---87.797--2021-08-09 13:35:00
-1.0---50.22--0.0---87.8--2021-08-09 13:36:00
-0.85---49.05-

####################
2021-08-10 16:34:00
RSI   SLOPE
60.37   1.95
********************
-0.38---62.51--2.12---87.969--2021-08-10 16:35:00
-0.89---65.08--2.63---87.997--2021-08-10 16:36:00
-0.63---62.87--1.29---87.983--2021-08-10 16:37:00
-0.76---63.5--0.73---87.99--2021-08-10 16:38:00
-0.83---63.86--0.7---87.994--2021-08-10 16:39:00
-0.74---63.06--0.11---87.989--2021-08-10 16:40:00
-0.96---64.17---0.18---88.001--2021-08-10 16:41:00
-1.07---64.71--0.37---88.007--2021-08-10 16:42:00
-1.16---65.16--0.33---88.012--2021-08-10 16:43:00
-1.0---63.66---0.04---88.003--2021-08-10 16:44:00
-0.78---61.71---0.27---87.991--2021-08-10 16:45:00
-0.71---61.08---0.62---87.987--2021-08-10 16:46:00
-0.56---59.81--0.0---87.979--2021-08-10 16:47:00
-1.16---63.05---0.42---88.012--2021-08-10 16:48:00
-1.21---63.33---0.07---88.015--2021-08-10 16:49:00
-1.3---63.79--0.42---88.02--2021-08-10 16:50:00
-1.38---64.17--0.62---88.024--2021-08-10 16:51:00
-1.5---64.81--1.0---88.031--2021-08-10 16:52:00
-1.74---65.98--0

In [26]:
sum(profit)

2.8

In [27]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->0
Total negative -->0
Total positive sm -->2.8
Total positive -->2
Length 2


In [ ]:
# a['rsi'].plot(figsize=(15,6))
# rates_frame['ll'] = rates_frame['close'].rolling(window=100).mean()
a['slope'].plot(figsize=(15,6))